# Task 8 · Receipts, Refunds & Reconciliation

## Spend-Quality Guardrail

### Objective

This notebook implements a Spend-Quality Guardrail for a Pay-per-Application platform.

The goal is to prevent students from spending money on low-quality job matches by introducing confidence-based recommendations.

The notebook demonstrates:

- Loading real datasets
- Job matching
- Spend-quality guardrail
- Low-fit warning
- Live verification
- Quantitative evaluation
- Edge-case handling
- Business interpretation

# 1. Import Libraries

The following libraries are used for:

- Data processing
- Similarity computation
- Performance evaluation
- Result visualization

In [4]:
import pandas as pd
import numpy as np

from sklearn.metrics import confusion_matrix
from sklearn.metrics import precision_score
from sklearn.metrics import recall_score

pd.set_option("display.max_columns",None)
pd.set_option("display.width",120)

# 2. Load Datasets

Three datasets are used in this notebook.

**students.csv**

Contains student information.

**jobs.csv**

Contains available job opportunities.

**matches.csv**

Contains the baseline matching scores generated in previous tasks.

In [6]:
students = pd.read_csv("../datasets/students.csv")

jobs = pd.read_csv("../datasets/jobs.csv")

matches = pd.read_csv("../datasets/matches.csv")

In [7]:
print("="*60)
print("Students Dataset")
print("="*60)

display(students.head())

print()

print("="*60)
print("Jobs Dataset")
print("="*60)

display(jobs.head())

print()

print("="*60)
print("Matches Dataset")
print("="*60)

display(matches.head())

Students Dataset


,student_id,skills,internship_months,education_level,certifications,preferred_role,location
0,1,"Python:85,SQL:75,Excel:70,Pandas:80",18,BTech,"Python,SQL",Data Analyst,Pune
1,2,"Java:80,Spring:75,SQL:65,Git:70",24,BE,Java,Backend Developer,Mumbai
2,3,"Python:90,ML:85,TensorFlow:75,SQL:70",12,MCA,ML,ML Engineer,Bangalore
3,4,"Excel:85,SQL:60,PowerBI:80",14,BTech,PowerBI,BI Analyst,Pune
4,5,"JavaScript:85,React:80,HTML:90,CSS:85",16,BE,Web,Frontend Developer,Hyderabad



Jobs Dataset


,job_id,company_name,job_title,required_skills,min_experience_years,job_type,location
0,101,TechNova,Data Analyst,"Python:70,SQL:60,Excel:50",1,Hybrid,Pune
1,102,CodeWorks,Backend Developer,"Java:70,Spring:65,SQL:60",2,Remote,Mumbai
2,103,AI Labs,ML Engineer,"Python:80,ML:70,TensorFlow:60",1,Hybrid,Bangalore
3,104,DataVision,BI Analyst,"Excel:70,SQL:60,PowerBI:70",1,Onsite,Pune
4,105,WebCraft,Frontend Developer,"JavaScript:70,React:70,HTML:70",1,Remote,Hyderabad



Matches Dataset


,student_id,job_id,skill_overlap_count,skill_overlap_ratio,experience_gap,label
0,1,101,3,1.000,2.0,1
1,1,102,1,0.333,1.0,0
2,1,103,1,0.333,2.0,0
3,1,104,2,0.667,2.0,1
4,1,105,0,0.000,2.0,0


In [8]:
print("Dataset Summary\n")

print("Students :", students.shape)
print("Jobs     :", jobs.shape)
print("Matches  :", matches.shape)

Dataset Summary

Students : (20, 7)
Jobs     : (9, 7)
Matches  : (180, 6)


In [9]:
print("\nMissing Values\n")

print(students.isnull().sum())

print()

print(jobs.isnull().sum())

print()

print(matches.isnull().sum())


Missing Values

student_id           0
skills               0
internship_months    0
education_level      0
certifications       1
preferred_role       0
location             0
dtype: int64

job_id                  0
company_name            0
job_title               0
required_skills         0
min_experience_years    0
job_type                0
location                0
dtype: int64

student_id             0
job_id                 0
skill_overlap_count    0
skill_overlap_ratio    0
experience_gap         0
label                  0
dtype: int64


# 3. Baseline Matching Results

The `matches.csv` dataset contains the baseline job-matching results generated in previous tasks.

Each record includes:

- Student ID
- Job ID
- Skill overlap count
- Skill overlap ratio
- Experience gap
- Ground-truth label

In this task, these baseline results are used to compute a confidence score before allowing students to pay for an application.

In [10]:
matches.head()

,student_id,job_id,skill_overlap_count,skill_overlap_ratio,experience_gap,label
0,1,101,3,1.000,2.0,1
1,1,102,1,0.333,1.0,0
2,1,103,1,0.333,2.0,0
3,1,104,2,0.667,2.0,1
4,1,105,0,0.000,2.0,0


# 4. Compute Match Confidence

A confidence score is calculated using:

- Skill Overlap Ratio (higher is better)
- Experience Gap (smaller is better)

Formula

Confidence Score =
0.7 × Skill Overlap Ratio
+
0.3 × Experience Compatibility

In [11]:
# Experience compatibility (0–1)

matches["experience_score"] = (
    matches["experience_gap"] /
    matches["experience_gap"].max()
)

matches["experience_score"] = 1 - matches["experience_score"]

# Final confidence score

matches["confidence_score"] = (
    0.7 * matches["skill_overlap_ratio"] +
    0.3 * matches["experience_score"]
)

matches["confidence_score"] = matches["confidence_score"].round(2)

display(matches.head())

,student_id,job_id,skill_overlap_count,skill_overlap_ratio,experience_gap,label,experience_score,confidence_score
0,1,101,3,1.000,2.0,1,0.6,0.88
1,1,102,1,0.333,1.0,0,0.8,0.47
2,1,103,1,0.333,2.0,0,0.6,0.41
3,1,104,2,0.667,2.0,1,0.6,0.65
4,1,105,0,0.000,2.0,0,0.6,0.18


# 5. Spend-Quality Guardrail

The guardrail classifies each recommendation into three categories.

| Confidence Score | Decision |
|------------------|----------|
| ≥ 0.75 | ALLOW |
| 0.60 – 0.74 | WARNING |
| < 0.60 | BLOCK |

This prevents students from spending money on poor-quality recommendations.

In [12]:
HIGH_CONFIDENCE = 0.75
WARNING_THRESHOLD = 0.60

def spend_quality_guardrail(score):

    if score >= HIGH_CONFIDENCE:
        return "ALLOW"

    elif score >= WARNING_THRESHOLD:
        return "WARNING"

    return "BLOCK"

In [13]:
matches["Decision"] = matches["confidence_score"].apply(
    spend_quality_guardrail
)

matches.head(10)

,student_id,job_id,skill_overlap_count,skill_overlap_ratio,experience_gap,label,experience_score,confidence_score,Decision
0,1,101,3,1.000,2.0,1,0.6,0.88,ALLOW
1,1,102,1,0.333,1.0,0,0.8,0.47,BLOCK
2,1,103,1,0.333,2.0,0,0.6,0.41,BLOCK
3,1,104,2,0.667,2.0,1,0.6,0.65,WARNING
4,1,105,0,0.000,2.0,0,0.6,0.18,BLOCK
5,1,106,0,0.000,1.0,0,0.8,0.24,BLOCK
6,1,107,0,0.000,1.0,0,0.8,0.24,BLOCK
7,1,108,0,0.000,2.0,0,0.6,0.18,BLOCK
8,1,109,1,0.333,1.0,0,0.8,0.47,BLOCK
9,2,101,1,0.333,3.0,0,0.4,0.35,BLOCK


# 6. Add Student & Job Details

To make the recommendations easier to interpret, the baseline matching results are merged with the student and job datasets.

This provides complete information including:

- Student ID
- Preferred Role
- Company Name
- Job Title
- Match Confidence
- Spend-Quality Decision

In [14]:
results = matches.merge(
    students[
        [
            "student_id",
            "preferred_role",
            "location"
        ]
    ],
    on="student_id"
)

results = results.merge(
    jobs[
        [
            "job_id",
            "company_name",
            "job_title"
        ]
    ],
    on="job_id"
)

display(results.head())

,student_id,job_id,skill_overlap_count,skill_overlap_ratio,experience_gap,label,experience_score,confidence_score,Decision,preferred_role,location,company_name,job_title
0,1,101,3,1.000,2.0,1,0.6,0.88,ALLOW,Data Analyst,Pune,TechNova,Data Analyst
1,1,102,1,0.333,1.0,0,0.8,0.47,BLOCK,Data Analyst,Pune,CodeWorks,Backend Developer
2,1,103,1,0.333,2.0,0,0.6,0.41,BLOCK,Data Analyst,Pune,AI Labs,ML Engineer
3,1,104,2,0.667,2.0,1,0.6,0.65,WARNING,Data Analyst,Pune,DataVision,BI Analyst
4,1,105,0,0.000,2.0,0,0.6,0.18,BLOCK,Data Analyst,Pune,WebCraft,Frontend Developer


# 7. Generate Spend-Quality Warning

Each recommendation is accompanied by an explanation.

This improves transparency and helps students understand why they are allowed, warned, or blocked before paying.

In [15]:
def generate_warning(decision):

    if decision == "ALLOW":
        return "✅ High-quality match. Safe to proceed with payment."

    elif decision == "WARNING":
        return "⚠️ Moderate match. Review the job description before paying."

    return "❌ Low-fit recommendation. Payment is not recommended."


results["Warning"] = results["Decision"].apply(generate_warning)

display(
    results[
        [
            "student_id",
            "company_name",
            "job_title",
            "confidence_score",
            "Decision",
            "Warning"
        ]
    ].head(10)
)

,student_id,company_name,job_title,confidence_score,Decision,Warning
0,1,TechNova,Data Analyst,0.88,ALLOW,✅ High-quality match. Safe to proceed with pay...
1,1,CodeWorks,Backend Developer,0.47,BLOCK,❌ Low-fit recommendation. Payment is not recom...
2,1,AI Labs,ML Engineer,0.41,BLOCK,❌ Low-fit recommendation. Payment is not recom...
3,1,DataVision,BI Analyst,0.65,WARNING,⚠️ Moderate match. Review the job description ...
4,1,WebCraft,Frontend Developer,0.18,BLOCK,❌ Low-fit recommendation. Payment is not recom...
5,1,CloudSphere,Cloud Engineer,0.24,BLOCK,❌ Low-fit recommendation. Payment is not recom...
6,1,SecureNet,Security Analyst,0.24,BLOCK,❌ Low-fit recommendation. Payment is not recom...
7,1,MobileWorks,Android Developer,0.18,BLOCK,❌ Low-fit recommendation. Payment is not recom...
8,1,SoftCore,Software Engineer,0.47,BLOCK,❌ Low-fit recommendation. Payment is not recom...
9,2,TechNova,Data Analyst,0.35,BLOCK,❌ Low-fit recommendation. Payment is not recom...


# 8. Live Verification

The Spend-Quality Guardrail is verified by counting:

- Allowed recommendations
- Warning recommendations
- Blocked recommendations

This demonstrates that the guardrail is working correctly on real matching data.

In [16]:
summary = results["Decision"].value_counts()

summary

Decision
BLOCK      162
ALLOW       12
WARNING      6
Name: count, dtype: int64

In [17]:
print("="*60)

print("LIVE VERIFICATION")

print("="*60)

print(f"Allowed Recommendations : {summary.get('ALLOW',0)}")

print(f"Warning Recommendations : {summary.get('WARNING',0)}")

print(f"Blocked Recommendations : {summary.get('BLOCK',0)}")

print("\nSpend-Quality Guardrail Verified Successfully.")

LIVE VERIFICATION
Allowed Recommendations : 12
Warning Recommendations : 6
Blocked Recommendations : 162

Spend-Quality Guardrail Verified Successfully.


# 9. Evaluation Metrics

To evaluate the Spend-Quality Guardrail, recommendations are converted into binary predictions.

- ALLOW → Positive Recommendation
- WARNING / BLOCK → Negative Recommendation

The following metrics are computed:

- Precision
- Recall
- False Positive Rate

In [18]:
results["prediction"] = (
    results["Decision"] == "ALLOW"
).astype(int)

precision = precision_score(
    results["label"],
    results["prediction"],
    zero_division=0
)

recall = recall_score(
    results["label"],
    results["prediction"],
    zero_division=0
)

cm = confusion_matrix(
    results["label"],
    results["prediction"]
)

tn, fp, fn, tp = cm.ravel()

fpr = fp / (fp + tn)

metrics = pd.DataFrame({

    "Metric":[
        "Precision",
        "Recall",
        "False Positive Rate"
    ],

    "Value":[
        round(precision,3),
        round(recall,3),
        round(fpr,3)
    ]

})

metrics

,Metric,Value
0,Precision,1.000
1,Recall,0.545
2,False Positive Rate,0.000


# 10. Business Interpretation

The Spend-Quality Guardrail ensures that students only spend money on high-confidence job matches.

Benefits:

- Reduces unnecessary spending.
- Filters weak recommendations.
- Improves recommendation quality.
- Increases trust in the matching system.
- Supports receipts, refunds, and reconciliation workflows.

In [19]:
print("="*60)

print("BUSINESS SUMMARY")

print("="*60)

print(f"Precision           : {precision:.2f}")

print(f"Recall              : {recall:.2f}")

print(f"False Positive Rate : {fpr:.2f}")

print()

if precision >= 0.80:
    print("✓ High-quality recommendations are being generated.")

if fpr < 0.30:
    print("✓ Low number of false positive paid applications.")

print("✓ Spend-Quality Guardrail successfully implemented.")

BUSINESS SUMMARY
Precision           : 1.00
Recall              : 0.55
False Positive Rate : 0.00

✓ High-quality recommendations are being generated.
✓ Low number of false positive paid applications.
✓ Spend-Quality Guardrail successfully implemented.


# 11. Edge Case Testing

The system should handle unusual situations gracefully.

The following scenarios are tested:

1. Very high confidence score.
2. Very low confidence score.
3. Boundary values.
4. Missing values.

In [20]:
test_scores = [1.0, 0.75, 0.60, 0.59, 0.20, np.nan]

print("="*60)
print("EDGE CASE TESTING")
print("="*60)

for score in test_scores:

    if pd.isna(score):
        decision = "BLOCK"
    else:
        decision = spend_quality_guardrail(score)

    print(f"Score : {score} --> {decision}")

EDGE CASE TESTING
Score : 1.0 --> ALLOW
Score : 0.75 --> ALLOW
Score : 0.6 --> WARNING
Score : 0.59 --> BLOCK
Score : 0.2 --> BLOCK
Score : nan --> BLOCK
